# Climate forcing for the 14 blocks (corrected, final)

**Alwar groundwater study · 14-block domain (LGD 574–587)**

**This is the climate notebook to use.** It produces the observed (IMD) and future (CMIP6, 13 GCMs, historical + four SSPs) block-daily rainfall, Tmax, Tmin and PET that feed the recharge model.

| | |
|---|---|
| **Reads** | `Vimal_Mishra_CMIP6/Copy of India.tar.xz`, the block weights from `01_cmip6_extraction_v1`, IMD rain NetCDF and Tmax/Tmin GRD, `Shapefile/LGD_Blocks.geojson` |
| **Writes** | `climate_forcing/Mishra_Alwar_CORRECTED/` (raw extraction + `QC/`), `inputs/alwar/climate_obs/`, `inputs/alwar/climate_cmip6/`, **`inputs/alwar/climate_cmip6_qdm/`** (final) |
| **Run time** | C2 about 75 min (reads the 29 GiB archive once); C9 about 13 min; the rest a few minutes each |

Run the cells in order. Every cell is self-contained (it defines its own paths), so you can restart from any cell once the files of the earlier cells exist. Methods and results: `docs/CLIMATE_FORCING_methods.md`.

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["geopandas", "xarray", "netCDF4", "pyarrow"], modflow=False)
del TOKEN, URL


## 1. Corrected grid-cell mapping (by longitude/latitude)
The weights file's `column_index_zero_based` is already the position of the cell in the raw file
(it counts the Year/Month/Day columns). The first extraction added 3 again and read cells ≈0.75° north of Alwar.
This cell locates every cell by the lon/lat stored in the file header and saves a corrected mapping.

In [ ]:
# ── C1: corrected grid-cell mapping by coordinates ──
import os
import numpy as np
import pandas as pd

GW = "/content/drive/MyDrive/gw_framework"
MISHRA = f"{GW}/climate_forcing/Vimal_Mishra_CMIP6"
SRC = f"{MISHRA}/SSP245_extracted/India"
SEL = f"{MISHRA}/SSP245_extracted/Alwar_grid_analysis/Area_weighted_spatial_selection"
WEIGHT_FILE = f"{SEL}/Alwar_block_grid_intersection_weights.csv"
FIXED_FILE = f"{SEL}/Alwar_block_grid_weights_CORRECTED.csv"

MODELS = ["ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CanESM5", "EC-Earth3",
          "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0", "MPI-ESM1-2-HR",
          "MPI-ESM1-2-LR", "MRI-ESM2-0", "NorESM2-LM", "NorESM2-MM"]
VARS = ["PrecipData", "TMaxData", "TMinData"]

def read_header(path):
    """lon row and lat row of a source file (all columns)."""
    h = pd.read_csv(path, sep=r"\s+", header=None, nrows=2)
    return h.iloc[0].to_numpy(float), h.iloc[1].to_numpy(float)

# 1. Reference header → (lon, lat) → raw column position
lon_row, lat_row = read_header(f"{SRC}/ACCESS-CM2/ssp245/TMaxData")
print("Columns in source file:", len(lon_row))
lookup = {(round(lon_row[p], 3), round(lat_row[p], 3)): p for p in range(3, len(lon_row))}

# 2. Locate each weights row by its own lon/lat
w = pd.read_csv(WEIGHT_FILE)
w["raw_col_position"] = [lookup.get((round(a, 3), round(b, 3)), -1)
                         for a, b in zip(w.longitude, w.latitude)]
nf = w[w.raw_col_position < 0]
print("Weights rows:", len(w), "| not found by coordinate:", len(nf))
w["pos_minus_old_index"] = w.raw_col_position - w.column_index_zero_based
print("raw_col_position - column_index_zero_based:")
print(w.pos_minus_old_index.value_counts().to_string())

# 3. Same grid in all 39 files? (ignore the 3 date columns: their header values are NaN)
diff = []
for m in MODELS:
    for v in VARS:
        lo, la = read_header(f"{SRC}/{m}/ssp245/{v}")
        same = (len(lo) == len(lon_row)
                and np.allclose(lo[3:], lon_row[3:])
                and np.allclose(la[3:], lat_row[3:]))
        if not same:
            diff.append((m, v, len(lo)))
print("Files with a different grid header:", len(diff), diff)

# 4. Corrected cells and weight sums
cells = w[w.raw_col_position >= 0].drop_duplicates("raw_col_position")
print("Distinct corrected cells:", len(cells),
      "| Lon", cells.longitude.min(), "-", cells.longitude.max(),
      "| Lat", cells.latitude.min(), "-", cells.latitude.max())
print("Weight sum per block (should be ~1):")
print(w.groupby("block_name").weight_within_intersected_area.sum().round(4).to_string())

# 5. Save (new file only)
if len(nf) == 0 and not diff:
    w["source_column"] = w["raw_col_position"]   # position in the raw file, incl. Year/Month/Day
    w.to_csv(FIXED_FILE, index=False)
    print("Saved:", FIXED_FILE)
else:
    print("NOT saved — fix the problems above first.")

## 2. One pass over the Mishra archive → Alwar cells and block-daily files
Reads `Copy of India.tar.xz` (29 GiB) once and writes, for 13 models × (historical + 4 SSPs) × 3 variables,
the 22 Alwar cells and the area-weighted 14-block series. About 75 minutes; resumable (finished files are skipped).

In [ ]:
# ── C2: one pass over the Mishra archive → correct Alwar cells + block-daily files ──
import os, time, shutil, subprocess, tarfile, traceback
import numpy as np
import pandas as pd

# stop an xz process left over from an earlier crashed run
try:
    proc.kill()
except NameError:
    pass

GW = "/content/drive/MyDrive/gw_framework"
MISHRA = f"{GW}/climate_forcing/Vimal_Mishra_CMIP6"
ARCHIVE = f"{MISHRA}/Copy of India.tar.xz"
FIXED_FILE = (f"{MISHRA}/SSP245_extracted/Alwar_grid_analysis/"
              "Area_weighted_spatial_selection/Alwar_block_grid_weights_CORRECTED.csv")
OUT = f"{GW}/climate_forcing/Mishra_Alwar_CORRECTED"
LOG = f"{OUT}/extraction_log.csv"
TMP = "/content/_tmp_member"          # local Colab disk, deleted after each file
os.makedirs(OUT, exist_ok=True)

SCENARIOS = ["historical", "ssp126", "ssp245", "ssp370", "ssp585"]
VARS = ["PrecipData", "TMaxData", "TMinData"]
TOTAL = 13 * len(SCENARIOS) * len(VARS)

# ── weights and the columns to read ──
w = pd.read_csv(FIXED_FILE)
cells = (w.drop_duplicates("source_column")
          .sort_values("source_column")[["source_column", "longitude", "latitude"]])
POS = cells.source_column.astype(int).tolist()
USECOLS = [0, 1, 2] + POS
CELL_NAMES = {p: f"cell_{lo:.3f}E_{la:.3f}N" for p, lo, la in
              zip(cells.source_column, cells.longitude, cells.latitude)}
EXP_LON = dict(zip(cells.source_column, cells.longitude))
EXP_LAT = dict(zip(cells.source_column, cells.latitude))

blocks = []
for (lgd, name), g in w.groupby(["block_lgd", "block_name"]):
    blocks.append((f"{lgd}_{name}", g.source_column.astype(int).tolist(),
                   g.weight_within_intersected_area.to_numpy(float)))
print(f"{len(POS)} grid cells, {len(blocks)} blocks")

log_rows = pd.read_csv(LOG).to_dict("records") if os.path.isfile(LOG) else []

def out_paths(model, scen, var):
    c = f"{OUT}/cells/{scen}/{model}_{scen}_{var}_cells.csv"
    b = f"{OUT}/blocks/{scen}/{model}_{scen}_{var}_AlwarBlocks.csv"
    return c, b

def safe_write(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def process_member(local_path):
    """Read the local copy of one archive file and build cell + block tables."""
    raw = pd.read_csv(local_path, sep=r"\s+", header=None, usecols=USECOLS, dtype=float)
    hdr, data = raw.iloc[:2], raw.iloc[2:].reset_index(drop=True)

    bad = [p for p in POS
           if abs(hdr.at[0, p] - EXP_LON[p]) > 1e-3 or abs(hdr.at[1, p] - EXP_LAT[p]) > 1e-3]
    if bad:
        raise ValueError(f"lon/lat mismatch at columns {bad[:5]}; "
                         f"file has lon={hdr.loc[0, bad[:3]].tolist()} lat={hdr.loc[1, bad[:3]].tolist()}")

    ymd = data[[0, 1, 2]].astype(int)
    ymd.columns = ["Year", "Month", "Day"]
    dates = pd.to_datetime(dict(year=ymd.Year, month=ymd.Month, day=ymd.Day), errors="coerce")

    cell_df = data[POS].rename(columns=CELL_NAMES)
    cell_out = pd.concat([dates.rename("Date"), ymd, cell_df], axis=1)

    blk_out = pd.concat([dates.rename("Date"), ymd], axis=1)
    for bname, bpos, bw in blocks:
        blk_out[bname] = data[bpos].to_numpy() @ bw
    return cell_out, blk_out, cell_df.to_numpy(), dates

# ── stream the archive (external xz is faster than Python's lzma) ──
assert shutil.which("xz"), "xz not found"
print("Archive:", ARCHIVE, f"({os.path.getsize(ARCHIVE)/2**30:.1f} GiB)")
proc = subprocess.Popen(["xz", "-dc", "-T0", ARCHIVE], stdout=subprocess.PIPE, bufsize=1 << 20)
tf = tarfile.open(fileobj=proc.stdout, mode="r|")

t0 = time.time(); done = skipped = failed = 0
try:
    for m in tf:
        if not m.isfile():
            continue
        parts = m.name.strip("/").split("/")          # India/<model>/<scenario>/<variable>
        if len(parts) != 4:
            continue
        _, model, scen, var = parts
        if scen not in SCENARIOS or var not in VARS:
            continue
        cpath, bpath = out_paths(model, scen, var)
        if os.path.isfile(cpath) and os.path.isfile(bpath):
            skipped += 1
            continue

        t1 = time.time()
        rec = {"model": model, "scenario": scen, "variable": var, "member": m.name,
               "size_GB": round(m.size / 1e9, 2)}
        try:
            # copy this one file from the stream to local disk, then read it normally
            with tf.extractfile(m) as src, open(TMP, "wb") as dst:
                shutil.copyfileobj(src, dst, length=16 << 20)
            cell_out, blk_out, vals, dates = process_member(TMP)
            safe_write(cell_out, cpath)
            safe_write(blk_out, bpath)
            rec.update(status="OK", rows=len(cell_out), invalid_dates=int(dates.isna().sum()),
                       first_date=str(dates.min().date()), last_date=str(dates.max().date()),
                       cell_min=float(np.nanmin(vals)), cell_max=float(np.nanmax(vals)),
                       n_nan=int(np.isnan(vals).sum()))
            done += 1
        except Exception as e:
            rec.update(status=f"FAILED: {type(e).__name__}: {e}",
                       trace=traceback.format_exc()[-1500:])
            failed += 1
        finally:
            if os.path.exists(TMP):
                os.remove(TMP)
        rec["seconds"] = round(time.time() - t1, 1)

        log_rows.append(rec)
        pd.DataFrame(log_rows).to_csv(LOG, index=False)
        print(f"[{done+failed+skipped:3d}/{TOTAL}] {model:14s} {scen:10s} {var:10s} "
              f"{rec['status'][:40]:40s} rows={rec.get('rows','-')} "
              f"{rec['seconds']}s | elapsed {(time.time()-t0)/60:.1f} min", flush=True)

        if rec["status"] != "OK":
            print("FULL ERROR:", rec["status"])
            print(rec["trace"])
        if failed >= 2 and done == 0:
            print("\nStopping early: the first files all failed.")
            break
finally:
    tf.close(); proc.stdout.close(); proc.kill(); proc.wait()

print(f"\nFINISHED: written {done}, skipped (already done) {skipped}, failed {failed}")
print("Total minutes:", round((time.time() - t0) / 60, 1))
print("Log:", LOG)

## 3. QC of the raw corrected files
Dates, calendars, missing values, Tmax < Tmin, and a climatology sanity table.
Result on 2026-11-03: 6 models use a no-leap calendar; 6,830 block-days with Tmax < Tmin;
889 temperatures outside −5…52 °C (traced in the next cell to 0 K fill values).

In [ ]:
# ── C3: QC of corrected Alwar CMIP6 block files (read-only; writes QC reports) ──
import os
import numpy as np
import pandas as pd

GW = "/content/drive/MyDrive/gw_framework"
OUT = f"{GW}/climate_forcing/Mishra_Alwar_CORRECTED"
QC = f"{OUT}/QC"; os.makedirs(QC, exist_ok=True)

MODELS = ["ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CanESM5", "EC-Earth3",
          "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0", "MPI-ESM1-2-HR",
          "MPI-ESM1-2-LR", "MRI-ESM2-0", "NorESM2-LM", "NorESM2-MM"]
SCENARIOS = ["historical", "ssp126", "ssp245", "ssp370", "ssp585"]
VARS = ["PrecipData", "TMaxData", "TMinData"]
PERIOD = {"historical": (1985, 2014)}                       # baseline
PERIOD.update({s: (2071, 2100) for s in SCENARIOS[1:]})    # far future

def load(model, scen, var):
    p = f"{OUT}/blocks/{scen}/{model}_{scen}_{var}_AlwarBlocks.csv"
    return pd.read_csv(p, parse_dates=["Date"])

file_rows, tmm_rows, clim_rows = [], [], []
for model in MODELS:
    for scen in SCENARIOS:
        data = {}
        for var in VARS:
            df = load(model, scen, var)
            bcols = [c for c in df.columns if c not in ("Date", "Year", "Month", "Day")]
            vals = df[bcols].to_numpy()
            d = df["Date"]
            full = pd.date_range(d.min(), d.max(), freq="D")
            missing = full.difference(d)
            rec = dict(model=model, scenario=scen, variable=var, rows=len(df),
                       first=str(d.min().date()), last=str(d.max().date()),
                       invalid_dates=int(d.isna().sum()),
                       duplicate_dates=int(d.duplicated().sum()),
                       missing_days=len(missing),
                       missing_only_feb29=bool(len(missing) > 0 and
                                               all((x.month, x.day) == (2, 29) for x in missing)),
                       has_feb29=bool(((d.dt.month == 2) & (d.dt.day == 29)).any()),
                       n_nan=int(np.isnan(vals).sum()),
                       vmin=round(float(np.nanmin(vals)), 2), vmax=round(float(np.nanmax(vals)), 2))
            if var == "PrecipData":
                rec["negative_values"] = int((vals < 0).sum())
            else:
                rec["outside_-5_52C"] = int(((vals < -5) | (vals > 52)).sum())
            file_rows.append(rec)
            data[var] = df.set_index("Date")[bcols]

        # Tmax < Tmin per block
        tx, tn = data["TMaxData"].align(data["TMinData"], join="inner")
        bad = (tx < tn).sum()
        for b, n in bad.items():
            tmm_rows.append(dict(model=model, scenario=scen, block=b, tmax_lt_tmin_days=int(n)))

        # climatology over the period (domain mean of the 14 blocks)
        y0, y1 = PERIOD[scen]
        sel = lambda x: x[(x.index.year >= y0) & (x.index.year <= y1)].mean(axis=1)
        p = sel(data["PrecipData"]); tmax = sel(data["TMaxData"]); tmin = sel(data["TMinData"])
        clim_rows.append(dict(model=model, scenario=scen, period=f"{y0}-{y1}",
                              P_mm_yr=round(p.groupby(p.index.year).sum().mean(), 0),
                              Tmax_C=round(tmax.mean(), 2), Tmin_C=round(tmin.mean(), 2)))
    print("checked", model, flush=True)

fq = pd.DataFrame(file_rows); tq = pd.DataFrame(tmm_rows); cq = pd.DataFrame(clim_rows)
fq.to_csv(f"{QC}/01_file_qc.csv", index=False)
tq.to_csv(f"{QC}/02_tmax_lt_tmin_by_block.csv", index=False)
cq.to_csv(f"{QC}/03_climatology_by_model.csv", index=False)

print("\n=== 1. FILE CHECKS (problems only) ===")
print("Files:", len(fq))
print("Invalid dates:", fq.invalid_dates.sum(), "| duplicate dates:", fq.duplicate_dates.sum(),
      "| NaN values:", fq.n_nan.sum())
print("Negative precipitation values:", int(fq.negative_values.fillna(0).sum()))
print("Temperatures outside -5..52 C:", int(fq["outside_-5_52C"].fillna(0).sum()))
gaps = fq[(fq.missing_days > 0) & ~fq.missing_only_feb29]
print("Files with gaps other than 29 Feb:", len(gaps))

print("\n=== 2. CALENDAR ===")
cal = (fq.groupby("model")
         .agg(has_feb29=("has_feb29", "all"), missing_days_max=("missing_days", "max"))
         .reset_index())
print(cal.to_string(index=False))

print("\n=== 3. Tmax < Tmin (block-days) ===")
print("Total:", int(tq.tmax_lt_tmin_days.sum()))
print(tq.groupby("model").tmax_lt_tmin_days.sum().to_string())

print("\n=== 4. CLIMATOLOGY (14-block mean) — historical 1985-2014 vs 2071-2100 ===")
for col, label in [("P_mm_yr", "Annual precipitation (mm/yr)"),
                   ("Tmax_C", "Mean Tmax (C)"), ("Tmin_C", "Mean Tmin (C)")]:
    print(f"\n{label}")
    print(cq.pivot(index="model", columns="scenario", values=col)[SCENARIOS].to_string())
print("\nQC files saved in:", QC)

In [ ]:
# ── C4: diagnose out-of-range temperatures and Tmax < Tmin (read-only) ──
import numpy as np
import pandas as pd

GW = "/content/drive/MyDrive/gw_framework"
OUT = f"{GW}/climate_forcing/Mishra_Alwar_CORRECTED"
QC = f"{OUT}/QC"
MODELS = ["ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CanESM5", "EC-Earth3",
          "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0", "MPI-ESM1-2-HR",
          "MPI-ESM1-2-LR", "MRI-ESM2-0", "NorESM2-LM", "NorESM2-MM"]
SCENARIOS = ["historical", "ssp126", "ssp245", "ssp370", "ssp585"]

def load(model, scen, var):
    df = pd.read_csv(f"{OUT}/blocks/{scen}/{model}_{scen}_{var}_AlwarBlocks.csv",
                     parse_dates=["Date"]).set_index("Date")
    return df.drop(columns=["Year", "Month", "Day"])

rng_rows, inv_rows = [], []
for model in MODELS:
    for scen in SCENARIOS:
        tx, tn = load(model, scen, "TMaxData"), load(model, scen, "TMinData")
        for var, df in (("TMaxData", tx), ("TMinData", tn)):
            v = df.to_numpy()
            lo, hi = (v < -5).sum(), (v > 52).sum()
            if lo or hi:
                rng_rows.append(dict(model=model, scenario=scen, variable=var,
                                     below_minus5=int(lo), above_52=int(hi),
                                     min=round(v.min(), 2), max=round(v.max(), 2)))
        tx, tn = tx.align(tn, join="inner")
        diff = (tn - tx).stack()                     # positive where Tmin > Tmax
        bad = diff[diff > 0]
        if len(bad):
            dates = bad.index.get_level_values(0)
            inv_rows.append(dict(model=model, scenario=scen, n=len(bad),
                                 median_gap_C=round(bad.median(), 2),
                                 p95_gap_C=round(bad.quantile(0.95), 2),
                                 max_gap_C=round(bad.max(), 2),
                                 top_month=int(pd.Series(dates.month).mode()[0]),
                                 share_after_2070=round((dates.year > 2070).mean(), 2)))
    print("done", model, flush=True)

rq = pd.DataFrame(rng_rows); iq = pd.DataFrame(inv_rows)
rq.to_csv(f"{QC}/04_out_of_range_temperatures.csv", index=False)
iq.to_csv(f"{QC}/05_tmax_lt_tmin_magnitude.csv", index=False)
print("\n=== OUT-OF-RANGE TEMPERATURES (-5..52 C) ===")
print(rq.to_string(index=False) if len(rq) else "none")
print("\n=== Tmax < Tmin: SIZE OF THE INVERSION (Tmin - Tmax, C) ===")
print(iq.sort_values("n", ascending=False).to_string(index=False))

## 4. Clean the cells → dataset-corrected block files
- values below −30 °C are 0 K fill codes (−273.15 °C): set to missing and interpolated in time, per cell;
- cell-days with Tmax < Tmin are swapped (artefact of the dataset's univariate bias correction);
- 29 February is added for the six no-leap models (temperature = mean of neighbours, rainfall = 0);
- blocks are recomputed from the cleaned cells.

Output: `inputs/alwar/climate_cmip6/` (the "dataset-only" version).

In [ ]:
# ── C5: clean cells (fill 0 K codes, swap Tmax<Tmin, add 29 Feb) → block files ──
import os
import numpy as np
import pandas as pd

GW = "/content/drive/MyDrive/gw_framework"
MISHRA = f"{GW}/climate_forcing/Vimal_Mishra_CMIP6"
OUT = f"{GW}/climate_forcing/Mishra_Alwar_CORRECTED"
CLEAN = f"{OUT}/cells_clean"
FINAL = f"{GW}/inputs/alwar/climate_cmip6"
QC = f"{OUT}/QC"
FIXED_FILE = (f"{MISHRA}/SSP245_extracted/Alwar_grid_analysis/"
              "Area_weighted_spatial_selection/Alwar_block_grid_weights_CORRECTED.csv")

MODELS = ["ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CanESM5", "EC-Earth3",
          "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0", "MPI-ESM1-2-HR",
          "MPI-ESM1-2-LR", "MRI-ESM2-0", "NorESM2-LM", "NorESM2-MM"]
SCENARIOS = ["historical", "ssp126", "ssp245", "ssp370", "ssp585"]
T_MISSING_BELOW = -30.0      # anything colder is a 0 K / fill code, not weather
P_MAX_VALID = 500.0          # mm/day; above this treat as fill code

# ── block weights keyed by cell column name ──
w = pd.read_csv(FIXED_FILE)
w["cell"] = [f"cell_{lo:.3f}E_{la:.3f}N" for lo, la in zip(w.longitude, w.latitude)]
BLOCKS = [(f"{lgd}_{name}", g.cell.tolist(), g.weight_within_intersected_area.to_numpy(float))
          for (lgd, name), g in w.groupby(["block_lgd", "block_name"])]

def read_cells(model, scen, var):
    df = pd.read_csv(f"{OUT}/cells/{scen}/{model}_{scen}_{var}_cells.csv", parse_dates=["Date"])
    return df.drop(columns=["Year", "Month", "Day"]).set_index("Date")

def longest_run(mask_col):
    """Longest run of True in a boolean Series."""
    m = mask_col.to_numpy()
    if not m.any():
        return 0
    runs = np.diff(np.flatnonzero(np.diff(np.r_[0, m.astype(int), 0])))[::2]
    return int(runs.max())

def fill_missing(df, mask):
    """Set masked values to NaN and fill by linear interpolation in time, per cell."""
    out = df.mask(mask)
    return out.interpolate(method="time", limit_direction="both")

def add_feb29(df, is_precip):
    """Insert 29 Feb rows: temperature = mean(28 Feb, 1 Mar); precipitation = 0."""
    full = pd.date_range(df.index.min(), df.index.max(), freq="D")
    missing = full.difference(df.index)
    if len(missing) == 0:
        return df, 0
    if not all((d.month, d.day) == (2, 29) for d in missing):
        raise ValueError(f"gaps other than 29 Feb: {missing[:5].tolist()}")
    out = df.reindex(full)
    if is_precip:
        out.loc[missing] = 0.0
    else:
        for d in missing:
            out.loc[d] = (df.loc[d - pd.Timedelta(days=1)] + df.loc[d + pd.Timedelta(days=1)]) / 2
    return out, len(missing)

def to_blocks(cells):
    out = pd.DataFrame(index=cells.index)
    for bname, bcells, bw in BLOCKS:
        out[bname] = cells[bcells].to_numpy() @ bw
    return out

def write(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    o = df.copy()
    o.insert(0, "Day", o.index.day); o.insert(0, "Month", o.index.month)
    o.insert(0, "Year", o.index.year); o.index.name = "Date"
    tmp = path + ".tmp"; o.reset_index().to_csv(tmp, index=False); os.replace(tmp, path)

log = []
for model in MODELS:
    for scen in SCENARIOS:
        p  = read_cells(model, scen, "PrecipData")
        tx = read_cells(model, scen, "TMaxData")
        tn = read_cells(model, scen, "TMinData")
        rec = dict(model=model, scenario=scen)

        # 1. fill codes
        mp = (p < 0) | (p > P_MAX_VALID)
        mx = tx < T_MISSING_BELOW
        mn = tn < T_MISSING_BELOW
        rec.update(p_filled=int(mp.values.sum()), tmax_filled=int(mx.values.sum()),
                   tmin_filled=int(mn.values.sum()),
                   longest_gap_days=max([longest_run(mx[c]) for c in mx] +
                                        [longest_run(mn[c]) for c in mn] +
                                        [longest_run(mp[c]) for c in mp]))
        p, tx, tn = fill_missing(p, mp), fill_missing(tx, mx), fill_missing(tn, mn)

        # 2. swap Tmax < Tmin at cell level
        inv = tx < tn
        rec["cell_days_swapped"] = int(inv.values.sum())
        tx_new, tn_new = tx.where(~inv, tn), tn.where(~inv, tx)
        tx, tn = tx_new, tn_new

        # 3. add 29 Feb where the model calendar has none
        p, n29 = add_feb29(p, True)
        tx, _ = add_feb29(tx, False)
        tn, _ = add_feb29(tn, False)
        rec["feb29_added"] = n29

        # 4. blocks and checks
        bp, bx, bn = to_blocks(p), to_blocks(tx), to_blocks(tn)
        rec.update(rows=len(bp), first=str(bp.index.min().date()), last=str(bp.index.max().date()),
                   nan_after=int(bp.isna().sum().sum() + bx.isna().sum().sum() + bn.isna().sum().sum()),
                   block_tmax_lt_tmin_after=int((bx < bn).values.sum()),
                   tmin_min_after=round(float(bn.values.min()), 2),
                   tmax_max_after=round(float(bx.values.max()), 2),
                   p_max_after=round(float(bp.values.max()), 1),
                   tmax_above_52=int((bx > 52).values.sum()))

        # 5. write cleaned cells and final blocks
        for var, c, b in (("PrecipData", p, bp), ("TMaxData", tx, bx), ("TMinData", tn, bn)):
            write(c, f"{CLEAN}/{scen}/{model}_{scen}_{var}_cells.csv")
            write(b, f"{FINAL}/{scen}/{model}_{scen}_{var}_AlwarBlocks.csv")
        log.append(rec)
    print("done", model, flush=True)

lg = pd.DataFrame(log)
lg.to_csv(f"{QC}/06_fix_log.csv", index=False)

print("\n=== FIXES APPLIED (totals) ===")
print(lg[["p_filled", "tmax_filled", "tmin_filled", "cell_days_swapped", "feb29_added"]].sum().to_string())
print("Longest filled gap (days):", lg.longest_gap_days.max())
print("\n=== CHECKS AFTER FIX (should be 0 / realistic) ===")
print("NaN remaining:", lg.nan_after.sum())
print("Block Tmax < Tmin remaining:", lg.block_tmax_lt_tmin_after.sum())
print("Lowest Tmin (C):", lg.tmin_min_after.min(), "| highest Tmax (C):", lg.tmax_max_after.max(),
      "| highest daily P (mm):", lg.p_max_after.max())
print("Block-days with Tmax > 52 C (kept, flagged):", lg.tmax_above_52.sum())
print("Row counts:", sorted(lg.rows.unique()))
print("\nFinal files in:", FINAL)

## 5. Observed climate (IMD) per block
IMD 0.25° rainfall 1951–2025 and IMD 1° Tmax/Tmin 1980–2024, area-weighted to the 14 blocks.
Also writes the small `blocks.geojson` and `domain.geojson` used by every later notebook.

In [ ]:
# ── C6: IMD observed rain / Tmax / Tmin → 14 Alwar blocks (area-weighted) ──
import os, glob, re, time
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from shapely.geometry import box

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
OBS = f"{ALW}/climate_obs"
os.makedirs(OBS, exist_ok=True)
LGD = list(range(574, 588))
UTM = "EPSG:32643"
RAIN_YEARS = range(1951, 2026)          # matches the GCM historical start
T_YEARS = range(1980, 2025)             # what is on Drive

# ── 1. Alwar blocks (read the big file once, save small copies) ──
t0 = time.time()
blk_path = f"{ALW}/blocks.geojson"
if os.path.isfile(blk_path):
    blocks = gpd.read_file(blk_path)
else:
    src = f"{GW}/Shapefile/LGD_Blocks.geojson"
    try:
        blocks = gpd.read_file(src, engine="pyogrio",
                               where=f"block_lgd IN ({','.join(map(str, LGD))})")
    except Exception:
        blocks = gpd.read_file(src)
    blocks["block_lgd"] = pd.to_numeric(blocks["block_lgd"], errors="coerce")
    blocks = blocks[blocks.block_lgd.isin(LGD)][["block_lgd", "block_name", "district", "geometry"]]
    blocks = blocks.to_crs("EPSG:4326").sort_values("block_lgd").reset_index(drop=True)
    blocks.to_file(blk_path, driver="GeoJSON")
    gpd.GeoDataFrame(geometry=[blocks.union_all()], crs="EPSG:4326").to_file(
        f"{ALW}/domain.geojson", driver="GeoJSON")
blocks["bname"] = blocks.block_lgd.astype(int).astype(str) + "_" + blocks.block_name
print(f"Blocks: {len(blocks)} | bounds {np.round(blocks.total_bounds, 3)} | {time.time()-t0:.0f}s")
assert len(blocks) == 14

def grid_weights(lats, lons, res, valid):
    """Area weights of grid cells (centres lats x lons, size res) in each block.
    valid: 2-D bool array (lat, lon) of cells that have data."""
    rows = []
    for i, la in enumerate(lats):
        for j, lo in enumerate(lons):
            if valid[i, j]:
                rows.append(dict(i=i, j=j, lat=la, lon=lo,
                                 geometry=box(lo - res/2, la - res/2, lo + res/2, la + res/2)))
    cells = gpd.GeoDataFrame(rows, crs="EPSG:4326").to_crs(UTM)
    b = blocks.to_crs(UTM)
    ix = gpd.overlay(b[["bname", "geometry"]], cells, how="intersection")
    ix["area"] = ix.area
    ix["w"] = ix.area / ix.groupby("bname").area.transform("sum")
    cover = (ix.groupby("bname").area.sum() / b.set_index("bname").area).rename("covered_fraction")
    return ix[["bname", "i", "j", "lat", "lon", "area", "w"]], cover

def to_blocks(arr, dates, wt):
    """arr: (time, nlat, nlon) → DataFrame of block values."""
    out = pd.DataFrame({"Date": dates})
    out["Year"], out["Month"], out["Day"] = dates.year, dates.month, dates.day
    for bname, g in wt.groupby("bname"):
        out[bname] = arr[:, g.i.to_numpy(), g.j.to_numpy()] @ g.w.to_numpy()
    return out

# ── 2. Rainfall 0.25° ──
LAT0, LAT1, LON0, LON1 = 26.5, 29.0, 75.5, 78.0
pieces = []
for y in RAIN_YEARS:
    with xr.open_dataset(f"{GW}/data/imd/netcdf/IMD_rain_0p25_{y}.nc") as ds:
        pieces.append(ds["rain"].sel(lat=slice(LAT0, LAT1), lon=slice(LON0, LON1)).load())
rain = xr.concat(pieces, "time")
r = rain.values.astype(float)
r[r < 0] = np.nan                                  # IMD missing code (-999) if present
valid_r = np.isfinite(r).mean(axis=0) > 0.99
wr, cov_r = grid_weights(rain.lat.values, rain.lon.values, 0.25, valid_r)
rain_blk = to_blocks(np.nan_to_num(r), pd.DatetimeIndex(rain.time.values), wr)
print(f"Rain: {rain_blk.Date.min().date()}–{rain_blk.Date.max().date()} | "
      f"{wr[['i','j']].drop_duplicates().shape[0]} cells | {time.time()-t0:.0f}s")

# ── 3. Temperature 1° (.GRD: 31x31 float32 per day, lat 7.5→37.5, lon 67.5→97.5) ──
TLAT = 7.5 + np.arange(31); TLON = 67.5 + np.arange(31)
def read_grd(var, y):
    f = [p for p in glob.glob(f"{GW}/data/imd_temp/{var}/*") if re.search(rf"{y}", os.path.basename(p))][0]
    a = np.fromfile(f, dtype="<f4")
    nd = a.size // (31 * 31)
    a = a.reshape(nd, 31, 31).astype(float)
    a[a > 90] = np.nan                              # IMD missing code 99.9
    return a, pd.date_range(f"{y}-01-01", periods=nd, freq="D")

T = {}
for var in ("tmax", "tmin"):
    arrs, dts = zip(*[read_grd(var, y) for y in T_YEARS])
    T[var] = (np.concatenate(arrs), pd.DatetimeIndex(np.concatenate(dts)))

# orientation check (Jan / May mean Tmax, 1981-2010)
tx, td = T["tmax"]
sel = (td.year >= 1981) & (td.year <= 2010)
def cell_mean(la, lo, month):
    i, j = int(la - 7.5), int(lo - 67.5)
    m = sel & (td.month == month)
    return round(float(np.nanmean(tx[m, i, j])), 1)
print("\nORIENTATION CHECK — mean Tmax (C), 1981-2010     Jan    May")
for name, la, lo in [("Alwar  27.5N 76.5E", 27.5, 76.5), ("Kerala 10.5N 76.5E", 10.5, 76.5),
                     ("Kashmir 33.5N 74.5E", 33.5, 74.5), ("Arabian Sea 15.5N 68.5E", 15.5, 68.5)]:
    print(f"  {name:26s} {cell_mean(la, lo, 1):6}  {cell_mean(la, lo, 5):6}")

valid_t = np.isfinite(T["tmax"][0]).mean(axis=0) > 0.99
wt_t, cov_t = grid_weights(TLAT, TLON, 1.0, valid_t)
tmax_blk = to_blocks(np.nan_to_num(T["tmax"][0]), T["tmax"][1], wt_t)
tmin_blk = to_blocks(np.nan_to_num(T["tmin"][0]), T["tmin"][1], wt_t)
print(f"\nTemp: {tmax_blk.Date.min().date()}–{tmax_blk.Date.max().date()} | "
      f"cells used: {wt_t[['lat','lon']].drop_duplicates().values.tolist()}")

# ── 4. Save ──
rain_blk.to_csv(f"{OBS}/IMD_rain_AlwarBlocks.csv", index=False)
tmax_blk.to_csv(f"{OBS}/IMD_tmax_AlwarBlocks.csv", index=False)
tmin_blk.to_csv(f"{OBS}/IMD_tmin_AlwarBlocks.csv", index=False)
wr.to_csv(f"{OBS}/weights_IMD_rain_0p25.csv", index=False)
wt_t.to_csv(f"{OBS}/weights_IMD_temp_1deg.csv", index=False)

# ── 5. QC summary ──
bc = [c for c in rain_blk.columns if c[:3].isdigit()]
def per(df, y0, y1):
    return df[(df.Year >= y0) & (df.Year <= y1)]
annual = per(rain_blk, 1985, 2014).groupby("Year")[bc].sum()
print("\n=== QC ===")
print("Block area covered by valid cells (should be ~1.0):",
      "rain", round(cov_r.min(), 3), "-", round(cov_r.max(), 3),
      "| temp", round(cov_t.min(), 3), "-", round(cov_t.max(), 3))
print("NaN in outputs:", int(rain_blk[bc].isna().sum().sum()),
      int(tmax_blk[bc].isna().sum().sum()), int(tmin_blk[bc].isna().sum().sum()))
print("Tmax < Tmin block-days:", int((tmax_blk[bc] < tmin_blk[bc]).values.sum()))
print(f"\nIMD 1985-2014, 14-block mean:  P = {annual.mean(axis=1).mean():.0f} mm/yr | "
      f"Tmax = {per(tmax_blk,1985,2014)[bc].values.mean():.2f} C | "
      f"Tmin = {per(tmin_blk,1985,2014)[bc].values.mean():.2f} C")
print("\nAnnual rainfall by block, 1985-2014 mean (mm):")
print(annual.mean().round(0).to_string())
print(f"\nSaved in {OBS} | total {time.time()-t0:.0f}s")

## 6. Hargreaves PET and evaluation of the GCMs against IMD (1985–2014)
PET uses Hargreaves–Samani for both IMD and CMIP6, because the GCM files hold only P, Tmax and Tmin.
Result: temperature and PET match IMD well, but several models put the rain in the wrong season.

In [ ]:
# ── C7: Hargreaves PET (IMD + all CMIP6) and GCM-vs-IMD evaluation 1985–2014 ──
import os, glob
import numpy as np
import pandas as pd
import geopandas as gpd

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
OBS = f"{ALW}/climate_obs"
CM = f"{ALW}/climate_cmip6"
QC = f"{GW}/climate_forcing/Mishra_Alwar_CORRECTED/QC"
MODELS = ["ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CanESM5", "EC-Earth3",
          "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0", "MPI-ESM1-2-HR",
          "MPI-ESM1-2-LR", "MRI-ESM2-0", "NorESM2-LM", "NorESM2-MM"]
SCENARIOS = ["historical", "ssp126", "ssp245", "ssp370", "ssp585"]
Y0, Y1 = 1985, 2014

# ── 0. the cleaning cell (C5) must have run ──
n_final = len(glob.glob(f"{CM}/*/*_AlwarBlocks.csv"))
if n_final < 195:
    raise RuntimeError(f"Only {n_final}/195 cleaned CMIP6 files found in {CM}. Run cell C5 first.")

# ── 1. Block centroid latitudes ──
blocks = gpd.read_file(f"{ALW}/blocks.geojson")
blocks["bname"] = blocks.block_lgd.astype(int).astype(str) + "_" + blocks.block_name
LAT = blocks.to_crs("EPSG:32643").centroid.to_crs("EPSG:4326").y
LAT = dict(zip(blocks.bname, LAT))

def ra_mm(doy, lat_deg):
    """FAO-56 extraterrestrial radiation (eq. 21), converted to mm/day of evaporation."""
    phi = np.radians(lat_deg)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    dec = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(dec), -1, 1))
    ra = (24 * 60 / np.pi) * 0.0820 * dr * (ws * np.sin(phi) * np.sin(dec)
                                             + np.cos(phi) * np.cos(dec) * np.sin(ws))
    return 0.408 * ra

def hargreaves(tx, tn):
    """tx, tn: DataFrames (Date, Year, Month, Day, blocks...). Returns PET in same layout."""
    out = tx[["Date", "Year", "Month", "Day"]].copy()
    doy = pd.to_datetime(tx.Date).dt.dayofyear.to_numpy()
    for b in [c for c in tx.columns if c[:3].isdigit()]:
        tmax, tmin = tx[b].to_numpy(), tn[b].to_numpy()
        dtr = np.clip(tmax - tmin, 0, None)
        out[b] = 0.0023 * ra_mm(doy, LAT[b]) * ((tmax + tmin) / 2 + 17.8) * np.sqrt(dtr)
    return out

# ── 2. PET for IMD ──
itx = pd.read_csv(f"{OBS}/IMD_tmax_AlwarBlocks.csv")
itn = pd.read_csv(f"{OBS}/IMD_tmin_AlwarBlocks.csv")
ipet = hargreaves(itx, itn)
ipet.to_csv(f"{OBS}/IMD_pet_hs_AlwarBlocks.csv", index=False)
BC = [c for c in ipet.columns if c[:3].isdigit()]

# ── 3. PET for every CMIP6 run ──
for model in MODELS:
    for scen in SCENARIOS:
        tx = pd.read_csv(f"{CM}/{scen}/{model}_{scen}_TMaxData_AlwarBlocks.csv")
        tn = pd.read_csv(f"{CM}/{scen}/{model}_{scen}_TMinData_AlwarBlocks.csv")
        hargreaves(tx, tn).to_csv(f"{CM}/{scen}/{model}_{scen}_PET_HS_AlwarBlocks.csv", index=False)
    print("PET done", model, flush=True)

# ── 4. GCM historical vs IMD, 1985-2014, 14-block mean ──
def summary(p, tx, tn, pet):
    sel = lambda d: d[(d.Year >= Y0) & (d.Year <= Y1)]
    p, tx, tn, pet = map(sel, (p, tx, tn, pet))
    dm = lambda d: d[BC].mean(axis=1)
    pd_ = dm(p)
    ann = pd_.groupby(p.Year.values).sum().mean()
    jjas = pd_[p.Month.isin([6, 7, 8, 9]).values].sum() / pd_.sum()
    return dict(P_mm_yr=ann, JJAS_share=jjas,
                wet_days_yr=(pd_ > 1).groupby(p.Year.values).sum().mean(),
                Tmax=dm(tx).mean(), Tmin=dm(tn).mean(),
                PET_mm_yr=dm(pet).groupby(pet.Year.values).sum().mean())

ip = pd.read_csv(f"{OBS}/IMD_rain_AlwarBlocks.csv")
ref = summary(ip, itx, itn, ipet)
rows = [dict(model="IMD (observed)", **ref)]
for model in MODELS:
    f = lambda v: pd.read_csv(f"{CM}/historical/{model}_historical_{v}_AlwarBlocks.csv")
    rows.append(dict(model=model, **summary(f("PrecipData"), f("TMaxData"), f("TMinData"), f("PET_HS"))))
ev = pd.DataFrame(rows).set_index("model")
ev["P_bias_%"] = 100 * (ev.P_mm_yr / ref["P_mm_yr"] - 1)
ev["Tmax_bias"] = ev.Tmax - ref["Tmax"]
ev["Tmin_bias"] = ev.Tmin - ref["Tmin"]
ev["PET_bias_%"] = 100 * (ev.PET_mm_yr / ref["PET_mm_yr"] - 1)
ev.round(2).to_csv(f"{QC}/07_gcm_vs_imd_1985_2014.csv")

pd.set_option("display.width", 200)
print(f"\n=== GCM historical vs IMD, {Y0}-{Y1}, 14-block mean ===")
print(ev[["P_mm_yr", "P_bias_%", "JJAS_share", "wet_days_yr", "Tmax", "Tmax_bias",
          "Tmin", "Tmin_bias", "PET_mm_yr", "PET_bias_%"]].round(2).to_string())

In [ ]:
# ── C8: is the seasonal cycle shifted? (read-only diagnosis) ──
import numpy as np
import pandas as pd

GW = "/content/drive/MyDrive/gw_framework"
OBS = f"{GW}/inputs/alwar/climate_obs"
CM = f"{GW}/inputs/alwar/climate_cmip6"
MODELS = ["ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CanESM5", "EC-Earth3",
          "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0", "MPI-ESM1-2-HR",
          "MPI-ESM1-2-LR", "MRI-ESM2-0", "NorESM2-LM", "NorESM2-MM"]

def clim(path, y0, y1, how):
    """Monthly climatology of the 14-block mean (sum per month for P, mean for T)."""
    d = pd.read_csv(path)
    d = d[(d.Year >= y0) & (d.Year <= y1)]
    bc = [c for c in d.columns if c[:3].isdigit()]
    s = d[bc].mean(axis=1)
    if how == "sum":
        return s.groupby([d.Year.values, d.Month.values]).sum().groupby(level=1).mean()
    return s.groupby(d.Month.values).mean()

def best_shift(model_c, ref_c):
    """Circular month shift (−6..+5) that best correlates the model cycle with IMD."""
    best = max(range(-6, 6), key=lambda k: np.corrcoef(np.roll(model_c.values, k), ref_c.values)[0, 1])
    return best, round(np.corrcoef(model_c.values, ref_c.values)[0, 1], 2)

ref = {"P": clim(f"{OBS}/IMD_rain_AlwarBlocks.csv", 1985, 2014, "sum"),
       "Tmax": clim(f"{OBS}/IMD_tmax_AlwarBlocks.csv", 1985, 2014, "mean"),
       "Tmin": clim(f"{OBS}/IMD_tmin_AlwarBlocks.csv", 1985, 2014, "mean")}
VARMAP = {"P": ("PrecipData", "sum"), "Tmax": ("TMaxData", "mean"), "Tmin": ("TMinData", "mean")}
rows = []
for m in MODELS:
    rec = dict(model=m)
    for v, (fv, how) in VARMAP.items():
        c = clim(f"{CM}/historical/{m}_historical_{fv}_AlwarBlocks.csv", 1985, 2014, how)
        k, r0 = best_shift(c, ref[v])
        rec[f"{v}_peak"] = int(c.idxmin() if v == "Tmin" else c.idxmax())
        rec[f"{v}_r"] = r0
        rec[f"{v}_shift"] = k
    p = clim(f"{CM}/historical/{m}_historical_PrecipData_AlwarBlocks.csv", 1985, 2014, "sum")
    rec["JJAS_share"] = round(p.loc[6:9].sum() / p.sum(), 2)
    rows.append(rec)
res = pd.DataFrame(rows)
pd.set_option("display.width", 220)
print("IMD peak P month:", ref["P"].idxmax(), "| hottest Tmax month:", ref["Tmax"].idxmax(),
      "| coldest Tmin month:", ref["Tmin"].idxmin())
print(res.to_string(index=False))
res.to_csv(f"{GW}/climate_forcing/Mishra_Alwar_CORRECTED/QC/08_seasonal_cycle_check.csv", index=False)

## 7. Monthly QDM to IMD (final forcing)
Quantile delta mapping (Cannon et al. 2015) per block and calendar month, calibration 1985–2014,
30-year moving window per decade. Rain is multiplicative (wet threshold 0.1 mm, change ratio capped ×5);
temperature is corrected as mean temperature (additive) + diurnal range (multiplicative).

Output: `inputs/alwar/climate_cmip6_qdm/` — **use these files for the recharge model.**

In [ ]:
# ── C9: monthly QDM of all 13 GCMs to IMD (1985–2014) per block + Hargreaves PET ──
import os, time
import numpy as np
import pandas as pd
import geopandas as gpd

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
OBS = f"{ALW}/climate_obs"
SRC = f"{ALW}/climate_cmip6"            # dataset bias correction only
DST = f"{ALW}/climate_cmip6_qdm"        # + monthly QDM to IMD
QC = f"{GW}/climate_forcing/Mishra_Alwar_CORRECTED/QC"
os.makedirs(DST, exist_ok=True)

MODELS = ["ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CanESM5", "EC-Earth3",
          "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0", "MPI-ESM1-2-HR",
          "MPI-ESM1-2-LR", "MRI-ESM2-0", "NorESM2-LM", "NorESM2-MM"]
SSPS = ["ssp126", "ssp245", "ssp370", "ssp585"]
GOOD_MONSOON = ["BCC-CSM2-MR", "EC-Earth3", "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0",
                "MPI-ESM1-2-HR", "MPI-ESM1-2-LR", "NorESM2-MM"]   # raw JJAS share ≥0.75, r ≥0.86
CAL0, CAL1 = 1985, 2014
WET = 0.1          # mm/day dry/wet threshold
RATIO_MAX = 5.0    # cap on multiplicative change
rng = np.random.default_rng(42)

def read(path):
    d = pd.read_csv(path, parse_dates=["Date"]).set_index("Date")
    return d.drop(columns=["Year", "Month", "Day"])

def write(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    o = df.copy()
    o.insert(0, "Day", o.index.day); o.insert(0, "Month", o.index.month)
    o.insert(0, "Year", o.index.year); o.index.name = "Date"
    tmp = path + ".tmp"; o.reset_index().to_csv(tmp, index=False); os.replace(tmp, path)

def jitter(x):
    x = x.copy(); z = x < WET
    x[z] = rng.uniform(1e-4, WET, z.sum())
    return x

def ecdf_tau(sample, x):
    s = np.sort(sample); n = len(s)
    t = (np.searchsorted(s, x, "left") + np.searchsorted(s, x, "right")) / (2 * n)
    return np.clip(t, 0.5 / n, 1 - 0.5 / n)

def windows(y_first, y_last, t_first):
    """Decades to correct (from t_first) with a 30-yr window inside [y_first, y_last]."""
    out = []
    for d in range(t_first, y_last + 1, 10):
        t0, t1 = d, min(d + 9, y_last)
        w0, w1 = d - 10, d + 19
        if w0 < y_first: w0, w1 = y_first, y_first + 29
        if w1 > y_last:  w0, w1 = y_last - 29, y_last
        out.append((t0, t1, w0, w1))
    return out

stats = {"ratio_capped": 0, "ratio_total": 0}
def qdm_series(obs, modc, series, years, months, t_first, kind):
    """QDM for one block. obs/modc: (values, months) in the calibration period;
    series: full model series to correct; returns corrected array (NaN outside target)."""
    out = np.full(series.shape, np.nan)
    ov, om = obs; mv, mm = modc
    for t0, t1, w0, w1 in windows(years.min(), years.max(), t_first):
        for m in range(1, 13):
            o = ov[om == m]; c = mv[mm == m]
            win = series[(months == m) & (years >= w0) & (years <= w1)]
            tgt = (months == m) & (years >= t0) & (years <= t1)
            x = series[tgt]
            if kind == "mult":
                o, c, win, x = jitter(o), jitter(c), jitter(win), jitter(x)
            tau = ecdf_tau(win, x)
            qc, qo = np.quantile(c, tau), np.quantile(o, tau)
            if kind == "add":
                out[tgt] = qo + (x - qc)
            else:
                r = x / np.maximum(qc, 1e-4)
                stats["ratio_total"] += r.size; stats["ratio_capped"] += int((r > RATIO_MAX).sum())
                y = qo * np.clip(r, 0, RATIO_MAX)
                y[y < WET] = 0.0
                out[tgt] = y
    return out

# ── Hargreaves PET ──
blocks = gpd.read_file(f"{ALW}/blocks.geojson")
blocks["bname"] = blocks.block_lgd.astype(int).astype(str) + "_" + blocks.block_name
LAT = dict(zip(blocks.bname, blocks.to_crs("EPSG:32643").centroid.to_crs("EPSG:4326").y))
def ra_mm(doy, lat):
    phi = np.radians(lat); dr = 1 + 0.033 * np.cos(2*np.pi*doy/365)
    dec = 0.409 * np.sin(2*np.pi*doy/365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi)*np.tan(dec), -1, 1))
    return 0.408 * (24*60/np.pi) * 0.0820 * dr * (ws*np.sin(phi)*np.sin(dec) + np.cos(phi)*np.cos(dec)*np.sin(ws))
def hargreaves(tx, tn):
    doy = tx.index.dayofyear.to_numpy(); out = pd.DataFrame(index=tx.index)
    for b in tx.columns:
        out[b] = 0.0023 * ra_mm(doy, LAT[b]) * ((tx[b] + tn[b]) / 2 + 17.8) * np.sqrt(np.clip(tx[b] - tn[b], 0, None))
    return out

# ── observations, calibration period ──
def cal(df):
    return df[(df.index.year >= CAL0) & (df.index.year <= CAL1)]
oP = cal(read(f"{OBS}/IMD_rain_AlwarBlocks.csv"))
oX = cal(read(f"{OBS}/IMD_tmax_AlwarBlocks.csv")); oN = cal(read(f"{OBS}/IMD_tmin_AlwarBlocks.csv"))
oM, oD = (oX + oN) / 2, (oX - oN).clip(lower=0.1)
BC = list(oP.columns)

def correct_model(model):
    hist = {v: read(f"{SRC}/historical/{model}_historical_{v}_AlwarBlocks.csv")
            for v in ("PrecipData", "TMaxData", "TMinData")}
    hM = (hist["TMaxData"] + hist["TMinData"]) / 2
    hD = (hist["TMaxData"] - hist["TMinData"]).clip(lower=0.1)
    hP = hist["PrecipData"]
    runs = {"historical": (hP, hM, hD, 1951)}
    for s in SSPS:
        f = {v: read(f"{SRC}/{s}/{model}_{s}_{v}_AlwarBlocks.csv") for v in ("PrecipData", "TMaxData", "TMinData")}
        P = pd.concat([hP, f["PrecipData"]])
        M = pd.concat([hM, (f["TMaxData"] + f["TMinData"]) / 2])
        D = pd.concat([hD, (f["TMaxData"] - f["TMinData"]).clip(lower=0.1)])
        runs[s] = (P, M, D, 2015)

    for scen, (P, M, D, t_first) in runs.items():
        yrs, mon = P.index.year.to_numpy(), P.index.month.to_numpy()
        outP, outM, outD = (pd.DataFrame(index=P.index, columns=BC, dtype=float) for _ in range(3))
        for b in BC:
            cP, cM, cD = cal(hP)[b], cal(hM)[b], cal(hD)[b]
            cm = cP.index.month.to_numpy()
            outP[b] = qdm_series((oP[b].to_numpy(), oP.index.month.to_numpy()), (cP.to_numpy(), cm),
                                 P[b].to_numpy(), yrs, mon, t_first, "mult")
            outM[b] = qdm_series((oM[b].to_numpy(), oM.index.month.to_numpy()), (cM.to_numpy(), cm),
                                 M[b].to_numpy(), yrs, mon, t_first, "add")
            outD[b] = qdm_series((oD[b].to_numpy(), oD.index.month.to_numpy()), (cD.to_numpy(), cm),
                                 D[b].to_numpy(), yrs, mon, t_first, "mult")
        keep = P.index.year >= t_first
        p_, m_, d_ = outP[keep], outM[keep], outD[keep].clip(lower=0.1)
        tx, tn = m_ + d_ / 2, m_ - d_ / 2
        write(p_, f"{DST}/{scen}/{model}_{scen}_PrecipData_AlwarBlocks.csv")
        write(tx, f"{DST}/{scen}/{model}_{scen}_TMaxData_AlwarBlocks.csv")
        write(tn, f"{DST}/{scen}/{model}_{scen}_TMinData_AlwarBlocks.csv")
        write(hargreaves(tx, tn), f"{DST}/{scen}/{model}_{scen}_PET_HS_AlwarBlocks.csv")

t0 = time.time()
for model in MODELS:
    correct_model(model)
    print(f"QDM done {model:14s} | {(time.time()-t0)/60:.1f} min", flush=True)

pd.DataFrame({"model": MODELS, "good_monsoon": [m in GOOD_MONSOON for m in MODELS]}).to_csv(
    f"{DST}/model_subsets.csv", index=False)

# ── validation ──
def dm(path, y0, y1):
    d = read(path); d = d[(d.index.year >= y0) & (d.index.year <= y1)]
    return d.mean(axis=1)
def clim_p(s):
    return s.groupby([s.index.year, s.index.month]).sum().groupby(level=1).mean()
ref = clim_p(dm(f"{OBS}/IMD_rain_AlwarBlocks.csv", CAL0, CAL1))

rows = []
for model in MODELS:
    rec = {"model": model, "good_monsoon": model in GOOD_MONSOON}
    for tag, base in (("raw", SRC), ("qdm", DST)):
        ph = dm(f"{base}/historical/{model}_historical_PrecipData_AlwarBlocks.csv", CAL0, CAL1)
        pf = dm(f"{base}/ssp245/{model}_ssp245_PrecipData_AlwarBlocks.csv", 2071, 2100)
        xh = dm(f"{base}/historical/{model}_historical_TMaxData_AlwarBlocks.csv", CAL0, CAL1)
        xf = dm(f"{base}/ssp245/{model}_ssp245_TMaxData_AlwarBlocks.csv", 2071, 2100)
        c = clim_p(ph)
        rec[f"{tag}_P_hist"] = round(c.sum(), 0)
        rec[f"{tag}_JJAS"] = round(c.loc[6:9].sum() / c.sum(), 2)
        rec[f"{tag}_r_cycle"] = round(np.corrcoef(c.values, ref.values)[0, 1], 2)
        rec[f"{tag}_dP_ssp245_%"] = round(100 * (clim_p(pf).sum() / c.sum() - 1), 1)
        rec[f"{tag}_dTmax_ssp245"] = round(xf.mean() - xh.mean(), 2)
    rows.append(rec)
val = pd.DataFrame(rows)
val.to_csv(f"{QC}/09_qdm_validation.csv", index=False)

pd.set_option("display.width", 240)
print(f"\nIMD {CAL0}-{CAL1}: P = {ref.sum():.0f} mm/yr, JJAS share = {ref.loc[6:9].sum()/ref.sum():.2f}")
print(f"Rain change ratios capped at x{RATIO_MAX}: {stats['ratio_capped']:,} of {stats['ratio_total']:,}")
print("\n=== RAW vs QDM — historical 1985-2014 and change by 2071-2100 under SSP2-4.5 ===")
print(val.to_string(index=False))
print("Output:", DST, f"| total {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── C10: is the QDM rain-change inflation real or a method artefact? (read-only) ──
import numpy as np
import pandas as pd

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
OBS, RAW, QDM = f"{ALW}/climate_obs", f"{ALW}/climate_cmip6", f"{ALW}/climate_cmip6_qdm"
MODELS = ["ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CanESM5", "EC-Earth3",
          "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0", "MPI-ESM1-2-HR",
          "MPI-ESM1-2-LR", "MRI-ESM2-0", "NorESM2-LM", "NorESM2-MM"]
H0, H1, F0, F1, SCEN = 1985, 2014, 2071, 2100, "ssp245"

def dm(path, y0, y1):
    d = pd.read_csv(path, parse_dates=["Date"]).set_index("Date")
    d = d[(d.index.year >= y0) & (d.index.year <= y1)].drop(columns=["Year", "Month", "Day"])
    return d.mean(axis=1)
def monthly(s):   # mean monthly total (mm)
    return s.groupby([s.index.year, s.index.month]).sum().groupby(level=1).mean()

imd_m = monthly(dm(f"{OBS}/IMD_rain_AlwarBlocks.csv", H0, H1))
rows = []
for m in MODELS:
    rh = dm(f"{RAW}/historical/{m}_historical_PrecipData_AlwarBlocks.csv", H0, H1)
    rf = dm(f"{RAW}/{SCEN}/{m}_{SCEN}_PrecipData_AlwarBlocks.csv", F0, F1)
    qh = dm(f"{QDM}/historical/{m}_historical_PrecipData_AlwarBlocks.csv", H0, H1)
    qf = dm(f"{QDM}/{SCEN}/{m}_{SCEN}_PrecipData_AlwarBlocks.csv", F0, F1)
    rel = monthly(rf) / monthly(rh)                       # model's own monthly change factor
    expected = (imd_m * rel).sum() / imd_m.sum() - 1      # applied to IMD's seasons
    rows.append(dict(model=m,
                     raw_change_pct=round(100 * (rf.mean() / rh.mean() - 1), 1),
                     expected_monthly_delta_pct=round(100 * expected, 1),
                     qdm_change_pct=round(100 * (qf.mean() / qh.mean() - 1), 1),
                     max_monthly_factor=round(rel.max(), 2), month_of_max=int(rel.idxmax())))
res = pd.DataFrame(rows)
pd.set_option("display.width", 240)
print(f"=== Annual rain change, {SCEN} {F0}-{F1} vs {H0}-{H1} (14-block mean) ===")
print(res.to_string(index=False))
res.to_csv(f"{GW}/climate_forcing/Mishra_Alwar_CORRECTED/QC/10_qdm_change_check.csv", index=False)

## 8. Wet-day frequency adaptation (keeps the monthly totals)
QDM's dry-day treatment turns some dry days into drizzle. This step sets the number of wet days per
block/month/decade to IMD's frequency × the model's own relative change, removes the smallest excess wet
days and rescales the rest so each month-decade total is unchanged. The original QDM rain is backed up first.

In [ ]:
# ── C11: wet-day frequency adaptation of QDM rainfall (mass-preserving) ──
import os, shutil, time
import numpy as np
import pandas as pd

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
OBS, RAW, QDM = f"{ALW}/climate_obs", f"{ALW}/climate_cmip6", f"{ALW}/climate_cmip6_qdm"
BAK = f"{QDM}/_before_wetday_fix"
QC = f"{GW}/climate_forcing/Mishra_Alwar_CORRECTED/QC"
MODELS = ["ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CanESM5", "EC-Earth3",
          "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0", "MPI-ESM1-2-HR",
          "MPI-ESM1-2-LR", "MRI-ESM2-0", "NorESM2-LM", "NorESM2-MM"]
SSPS = ["ssp126", "ssp245", "ssp370", "ssp585"]
CAL0, CAL1, WET = 1985, 2014, 0.1

def read(path):
    d = pd.read_csv(path, parse_dates=["Date"]).set_index("Date")
    return d.drop(columns=["Year", "Month", "Day"])
def write(df, path):
    o = df.copy()
    o.insert(0, "Day", o.index.day); o.insert(0, "Month", o.index.month)
    o.insert(0, "Year", o.index.year); o.index.name = "Date"
    tmp = path + ".tmp"; o.reset_index().to_csv(tmp, index=False); os.replace(tmp, path)
def windows(y_first, y_last, t_first):      # same decades/windows as the QDM cell
    out = []
    for d in range(t_first, y_last + 1, 10):
        t0, t1, w0, w1 = d, min(d + 9, y_last), d - 10, d + 19
        if w0 < y_first: w0, w1 = y_first, y_first + 29
        if w1 > y_last:  w0, w1 = y_last - 29, y_last
        out.append((t0, t1, w0, w1))
    return out
def wetfreq(x):
    return (x >= WET).mean() if len(x) else np.nan

obs = read(f"{OBS}/IMD_rain_AlwarBlocks.csv")
obs = obs[(obs.index.year >= CAL0) & (obs.index.year <= CAL1)]
BC = list(obs.columns)
fo = {b: obs[b].groupby(obs.index.month).apply(wetfreq) for b in BC}     # IMD wet freq by month

log = []; t0 = time.time()
for model in MODELS:
    rawh = read(f"{RAW}/historical/{model}_historical_PrecipData_AlwarBlocks.csv")
    cal = rawh[(rawh.index.year >= CAL0) & (rawh.index.year <= CAL1)]
    fh = {b: cal[b].groupby(cal.index.month).apply(wetfreq) for b in BC}  # model hist wet freq
    for scen in ["historical"] + SSPS:
        qpath = f"{QDM}/{scen}/{model}_{scen}_PrecipData_AlwarBlocks.csv"
        bpath = f"{BAK}/{scen}/{os.path.basename(qpath)}"
        os.makedirs(os.path.dirname(bpath), exist_ok=True)
        if not os.path.isfile(bpath):
            shutil.copy2(qpath, bpath)               # keep the original QDM file once
        q = read(bpath)                              # always start from the original
        if scen == "historical":
            raw, t_first = rawh, 1951
        else:
            raw, t_first = pd.concat([rawh, read(f"{RAW}/{scen}/{model}_{scen}_PrecipData_AlwarBlocks.csv")]), 2015
        ry, rm = raw.index.year.to_numpy(), raw.index.month.to_numpy()
        qy, qm = q.index.year.to_numpy(), q.index.month.to_numpy()
        out = q.copy()
        removed = wet_before = 0
        for b in BC:
            rv, qv = raw[b].to_numpy(), out[b].to_numpy().copy()
            for d0, d1, w0, w1 in windows(ry.min(), ry.max(), t_first):
                for m in range(1, 13):
                    fp = wetfreq(rv[(rm == m) & (ry >= w0) & (ry <= w1)])
                    ft = fo[b][m] * (fp / fh[b][m]) if fh[b][m] > 0 else fo[b][m]
                    ft = min(max(ft, 0.0), 1.0)
                    sel = np.flatnonzero((qm == m) & (qy >= d0) & (qy <= d1))
                    if sel.size == 0:
                        continue
                    x = qv[sel]
                    wet = np.flatnonzero(x >= WET)
                    wet_before += wet.size
                    k = int(round(ft * sel.size))
                    if wet.size > k:
                        total = x.sum()
                        order = wet[np.argsort(x[wet])]          # smallest wet values first
                        drop = order[: wet.size - k]
                        x[drop] = 0.0
                        keep_sum = x.sum()
                        if keep_sum > 0:
                            x *= total / keep_sum                # preserve the month-decade total
                        removed += drop.size
                        qv[sel] = x
            out[b] = qv
        write(out, qpath)
        log.append(dict(model=model, scenario=scen, wet_days_removed=removed,
                        share_removed=round(removed / max(wet_before, 1), 3),
                        total_before=round(q.values.sum(), 1), total_after=round(out.values.sum(), 1)))
    print(f"done {model:14s} | {(time.time()-t0)/60:.1f} min", flush=True)

lg = pd.DataFrame(log)
lg["total_change_pct"] = (100 * (lg.total_after / lg.total_before - 1)).round(4)
lg.to_csv(f"{QC}/11_wetday_fix_log.csv", index=False)

# ── validation: wet days per year and change signal ──
def dm(path, y0, y1):
    d = read(path); d = d[(d.index.year >= y0) & (d.index.year <= y1)]
    return d.mean(axis=1)
def wetdays(path, y0, y1):
    d = read(path); d = d[(d.index.year >= y0) & (d.index.year <= y1)]
    return (d >= WET).groupby(d.index.year).sum().mean().mean()

imd_wd = (obs >= WET).groupby(obs.index.year).sum().mean().mean()
rows = []
for m in MODELS:
    h = f"{QDM}/historical/{m}_historical_PrecipData_AlwarBlocks.csv"
    f = f"{QDM}/ssp245/{m}_ssp245_PrecipData_AlwarBlocks.csv"
    rh = f"{RAW}/historical/{m}_historical_PrecipData_AlwarBlocks.csv"
    rf = f"{RAW}/ssp245/{m}_ssp245_PrecipData_AlwarBlocks.csv"
    rows.append(dict(model=m,
                     wetdays_hist=round(wetdays(h, CAL0, CAL1), 1),
                     wetdays_fut=round(wetdays(f, 2071, 2100), 1),
                     raw_wetday_change_pct=round(100 * (wetdays(rf, 2071, 2100) / wetdays(rh, CAL0, CAL1) - 1), 1),
                     new_wetday_change_pct=round(100 * (wetdays(f, 2071, 2100) / wetdays(h, CAL0, CAL1) - 1), 1),
                     P_hist=round(dm(h, CAL0, CAL1).groupby(lambda x: x.year).sum().mean(), 0),
                     dP_ssp245_pct=round(100 * (dm(f, 2071, 2100).mean() / dm(h, CAL0, CAL1).mean() - 1), 1)))
val = pd.DataFrame(rows)
val.to_csv(f"{QC}/12_wetday_fix_validation.csv", index=False)

pd.set_option("display.width", 220)
print(f"\nWet days removed: {lg.wet_days_removed.sum():,} | max change in totals: "
      f"{lg.total_change_pct.abs().max():.4f} %")
print(f"IMD wet days/yr ({CAL0}-{CAL1}, block basis): {imd_wd:.1f}")
print(val.to_string(index=False))
print("Backup of original QDM rain:", BAK)